# ANA 500 — Module 3 Micro-Project 3
**Adult Income Classification | Logistic Regression vs Support Vector Machine**  
**Shawn Jones — October 2026**

**Research question:** Which education and work characteristics are associated with annual income >$50K?

**Hypothesis:** Higher educational attainment, additional hours worked, and selected occupations/work classes are associated with a higher proportion of adults earning >$50K. This is an association study, not a causal study.

**Continuity:** Builds on Week 2 prepared Adult dataset (32,537 rows, eight columns). Models use the identical holdout set, suitable for future Week 4 comparisons. Results below are calculated from the actual dataset when this notebook runs; they have not been fabricated.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, ConfusionMatrixDisplay, classification_report)

SEED = 42
OUT = Path('week3_outputs'); OUT.mkdir(exist_ok=True)
print('Libraries loaded; results folder:', OUT.resolve())

## 1. Data acquisition and validation
Place the file `adult_income_prepared.csv` from Week 2 in the **same folder** as this notebook. No example rows are substituted. A missing file produces an explicit error.

In [ ]:
data_path = Path('adult_income_prepared.csv')
if not data_path.exists():
    raise FileNotFoundError('Missing adult_income_prepared.csv: copy your actual Week 2 prepared CSV into this notebook folder.')
df = pd.read_csv(data_path)
required = ['age','education','education_num','workclass','occupation',
            'hours_per_week','marital_status','income']
assert set(required).issubset(df.columns), f'Missing columns: {set(required)-set(df.columns)}'
df = df[required].copy()
for col in df.select_dtypes(include='object'):
    df[col] = df[col].astype('string').str.strip()
df = df.replace(['?', '', 'nan'], np.nan)
df['income'] = df['income'].str.replace('.', '', regex=False)
assert df['income'].notna().all() and set(df['income'].unique()) == {'<=50K','>50K'}, df['income'].value_counts(dropna=False)
for col in ['age','education_num','hours_per_week']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
print('Shape:',df.shape,'\nIncome counts:\n',df['income'].value_counts(),'\nMissing values:\n',df.isna().sum(),'\nDuplicates:',df.duplicated().sum())
print('Note: Week 2 mode imputation, if already applied, cannot be reversed.')

## 2. Exploratory analysis — directly addressing instructor feedback
All category comparisons calculate **within-category income proportions**, not raw category counts. Small groups should be interpreted cautiously.

In [ ]:
df['income_high'] = (df['income']=='>50K').astype(int)
def group_income_rate(feature):
    t = df.groupby(feature,dropna=False)['income_high'].agg(rate='mean',count='size').sort_values('rate')
    t['rate_pct'] = (100*t['rate']).round(1)
    return t

def rate_figure(feature, filename):
    t=group_income_rate(feature)
    fig,ax=plt.subplots(figsize=(10,max(4,0.34*len(t))))
    bars=ax.barh(t.index.astype(str), t['rate_pct'])
    ax.set(xlim=(0,100),xlabel='Percentage earning >$50K within group',title=f'Income >$50K by {feature}')
    for bar, count in zip(bars,t['count']):
        ax.text(bar.get_width()+0.7,bar.get_y()+bar.get_height()/2,f'n={count:,}',va='center',fontsize=8)
    ax.set_xlim(0,115);fig.tight_layout();fig.savefig(OUT/filename,dpi=175);plt.show()
    return t
for feat in ['education','occupation','workclass']:
    table=rate_figure(feat,f'income_rate_by_{feat}.png')
    print('\n',feat,'(lowest/highest groups; interpret with n):\n',table[['rate_pct','count']].to_string())
fig, ax = plt.subplots(figsize=(8,4.8))
groups=[df.loc[df.income==c,'hours_per_week'].dropna() for c in ['<=50K','>50K']]
ax.boxplot(groups,tick_labels=['<=50K','>50K'],showfliers=True)
ax.set(xlabel='Income class',ylabel='Hours per week',title='Weekly work hours by income class')
fig.tight_layout();fig.savefig(OUT/'hours_by_income.png',dpi=175);plt.show()
print(df.groupby('income')['hours_per_week'].agg(['count','mean','median','min','max']).round(2))

## 3. Modeling design and safeguards
Use one stratified 80/20 split. Avoid encoding `education` and `education_num` simultaneously because they represent the same qualification; use `education` to preserve interpretable categories. Missing predictor values are imputed *within each training pipeline*, preventing holdout leakage. Note: Week 2 had already imputed missing categorical data, which is a historical limitation.

Model set: majority baseline, logistic regression, linear-kernel support vector classifier. A linear SVM is a valid Support Vector Machine and generally more computationally efficient here than a nonlinear kernel.

In [ ]:
features = ['age','education','workclass','occupation','hours_per_week','marital_status']
X = df[features].copy(); y = df['income_high'].copy()
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.20,random_state=SEED,stratify=y)
num=['age','hours_per_week'];cat=['education','workclass','occupation','marital_status']
pre=ColumnTransformer([
    ('numeric',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),
    ('categorical',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),
          ('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
models = {
  'Majority baseline':Pipeline([('preprocess',pre),('classifier',DummyClassifier(strategy='most_frequent'))]),
  'Logistic regression':Pipeline([('preprocess',pre),('classifier',LogisticRegression(max_iter=2000,random_state=SEED))]),
  'Linear SVM':Pipeline([('preprocess',pre),('classifier',LinearSVC(max_iter=12000,random_state=SEED,dual=False))])}
print('Training:',len(X_train),'Test:',len(X_test),'High-income share:',round(y.mean(),4))

## 4. Cross-validation and identical holdout evaluation
Five-fold stratified training-only cross-validation is used for selection context. The test set is evaluated once per model; F1 and recall matter because the >$50K class is the minority.

In [ ]:
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
rows=[]
for name,model in models.items():
    score=cv_cross = cross_validate(model,X_train,y_train,cv=cv,scoring={'f1':'f1','accuracy':'accuracy'},n_jobs=1)
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    raw_score=model.decision_function(X_test) if hasattr(model,'decision_function') else (model.predict_proba(X_test)[:,1] if hasattr(model,'predict_proba') else None)
    result={'model':name,'cv_f1_mean':np.mean(score['test_f1']),'cv_f1_std':np.std(score['test_f1']),
       'accuracy':accuracy_score(y_test,pred),'precision':precision_score(y_test,pred,zero_division=0),
       'recall':recall_score(y_test,pred,zero_division=0),'f1':f1_score(y_test,pred,zero_division=0),
       'roc_auc':roc_auc_score(y_test,raw_score) if raw_score is not None else np.nan}
    rows.append(result)
    print('\n',name,'\n',classification_report(y_test,pred,target_names=['<=50K','>50K'],zero_division=0))
    fig,ax=plt.subplots(figsize=(4.5,4));ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=['<=50K','>50K'],ax=ax,cmap='Blues',colorbar=False)
    ax.set_title(name);fig.tight_layout();fig.savefig(OUT/f'confusion_{name.lower().replace(" ","_")}.png',dpi=175);plt.show()
results=pd.DataFrame(rows).set_index('model').sort_values('f1',ascending=False)
print('\nTest metrics (verified by executing this notebook):\n',results.round(4).to_string())
results.to_csv(OUT/'model_comparison.csv')
print('Best test F1:',results.index[0])

## 5. Interpretation, limits and decision
Run the next cell to generate concise observations using **actual measured** rates and scores. Do not assert causation, fairness, or real-world deployment suitability solely from these results.

In [ ]:
ed=group_income_rate('education'); occupation=group_income_rate('occupation'); wc=group_income_rate('workclass')
print(f'High-income proportion overall: {y.mean():.1%}.')
for name,t in [('education',ed),('occupation',occupation),('workclass',wc)]:
    sizable=t[t['count']>=100]
    print(f'Among {name} groups with at least 100 records: lowest is {sizable.index[0]} ({sizable.iloc[0]["rate"]:.1%}, n={int(sizable.iloc[0]["count"])}), highest is {sizable.index[-1]} ({sizable.iloc[-1]["rate"]:.1%}, n={int(sizable.iloc[-1]["count"])}).')
print('Median weekly hours:',df.groupby('income')['hours_per_week'].median().to_dict())
print('Strongest held-out F1:',results.index[0],round(results.iloc[0]['f1'],3))
print('Limitations: observational/historical US census records; earlier imputation; omitted variables; label imbalance; possible social bias; no causal conclusions.')
print('Week 4: compare neural network on the same fixed holdout and comparable F1/precision/recall/ROC-AUC.')

## References
Dua, D., & Graff, C. (2019). *UCI Machine Learning Repository: Adult* [Data set]. University of California, Irvine. https://archive.ics.uci.edu/dataset/2/adult

Scikit-learn developers. (n.d.). *Scikit-learn user guide*. https://scikit-learn.org/stable/user_guide.html

National University. (2026). *ANA 500: Module 3 logistic regression and SVM tutorial* [Course material; verify exact author and title against the supplied PDF].

**Submission verification:** Ensure notebook runs top-to-bottom without errors; include generated `week3_outputs` charts and `model_comparison.csv` in slides; check course rubric and add GitHub URL if required. No model metrics should be reported until execution on the original CSV.